# Chapter 11 Lab — Human-in-the-Loop

**Book:** Agentic AI: Build, Ship, Portfolio  
**Chapter:** 11 — Human-in-the-Loop  
**Goal:** Build agents that know when to pause, ask for approval, and escalate to humans — keeping autonomy where it helps and oversight where it matters.

### What you will do in this lab

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | The Case for Human Oversight |
| 3 | Interrupt Points |
| 4 | Approval Gates |
| 5 | Confidence Thresholds |
| 6 | Escalation Patterns |
| 7 | UX for Human Review |
| 8 | Resuming After Interruption |
| 9 | LangGraph `interrupt()` |
| 10 | Exercises |

> **Prerequisites:** An OpenAI API key stored in a `.env` file as `OPENAI_API_KEY`.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
# Run once to install required packages (restart the kernel afterwards).

%pip install langgraph langchain-openai langchain-core openai python-dotenv --quiet

In [ ]:
import json
import os
import textwrap
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from typing import Any, Optional

from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY from .env

import openai

client = openai.OpenAI()

# Quick connectivity check
resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Say 'ready' in one word."}],
    max_tokens=5,
)
print("API check:", resp.choices[0].message.content)

---
## 2. The Case for Human Oversight

Autonomous agents are powerful but fallible. Here are real failure modes that demand human oversight:

| Failure Mode | Example | Consequence |
|-------------|---------|-------------|
| **Hallucinated action** | Agent invents a customer refund policy | Financial loss |
| **Irreversible side-effect** | Agent deletes a production database table | Data loss |
| **Ambiguous intent** | "Cancel my subscription" — which one? | Wrong cancellation |
| **Compliance violation** | Agent shares PII in a response | Regulatory fine |
| **Confidence collapse** | Agent guesses at a medical dosage | Patient harm |

The pattern is clear: **agents should pause and ask when the stakes are high or the confidence is low.**

Let's see this in action.

In [ ]:
# ── Demonstrating agent mistakes that need human review ────────────────────

risky_scenarios = [
    {
        "task": "Process a $50,000 wire transfer to account ending 4829",
        "risk": "HIGH",
        "reason": "Large financial transaction — irreversible once sent",
    },
    {
        "task": "Delete all records older than 30 days from the users table",
        "risk": "HIGH",
        "reason": "Bulk data deletion — cannot be undone without backup",
    },
    {
        "task": "Reply to the customer: 'Your account password is hunter42'",
        "risk": "CRITICAL",
        "reason": "Exposing credentials in plain text — compliance violation",
    },
    {
        "task": "Schedule a meeting for next Tuesday at 3pm",
        "risk": "LOW",
        "reason": "Easily reversible, low-stakes action",
    },
    {
        "task": "Recommend dosage of 500mg ibuprofen every 4 hours",
        "risk": "HIGH",
        "reason": "Medical advice — agent is not a licensed provider",
    },
]

print("Agent Risk Assessment")
print("=" * 70)
for s in risky_scenarios:
    emoji = {"LOW": "[OK]", "HIGH": "[!!]", "CRITICAL": "[XX]"}[s["risk"]]
    action = "PROCEED" if s["risk"] == "LOW" else "PAUSE FOR HUMAN REVIEW"
    print(f"\n{emoji} Risk: {s['risk']}  |  Action: {action}")
    print(f"   Task:   {s['task']}")
    print(f"   Reason: {s['reason']}")

### Key insight

Not every action needs a human. The art is in **classifying risk** and routing only what matters. The sections that follow build this machinery piece by piece.

---
## 3. Interrupt Points

An **interrupt point** is a checkpoint in the agent loop where execution pauses and control returns to the caller. The simplest version: after every tool call, ask "continue or stop?"

We will build a minimal agent loop with configurable interrupt points.

In [ ]:
# ── Simulated tools for the agent ─────────────────────────────────────────

def lookup_account(account_id: str) -> dict:
    """Simulate looking up an account."""
    accounts = {
        "ACC-001": {"name": "Alice Johnson", "balance": 15_420.50, "status": "active"},
        "ACC-002": {"name": "Bob Smith", "balance": 87_300.00, "status": "active"},
        "ACC-003": {"name": "Carol White", "balance": 230.15, "status": "frozen"},
    }
    return accounts.get(account_id, {"error": f"Account {account_id} not found"})


def transfer_funds(from_acct: str, to_acct: str, amount: float) -> dict:
    """Simulate a fund transfer."""
    return {
        "status": "completed",
        "from": from_acct,
        "to": to_acct,
        "amount": amount,
        "confirmation": f"TXN-{uuid.uuid4().hex[:8].upper()}",
    }


def send_email(to: str, subject: str, body: str) -> dict:
    """Simulate sending an email."""
    return {"status": "sent", "to": to, "subject": subject}


TOOLS = {
    "lookup_account": lookup_account,
    "transfer_funds": transfer_funds,
    "send_email": send_email,
}

print("Tools registered:", list(TOOLS.keys()))

In [ ]:
# ── Agent loop with interrupt points ──────────────────────────────────────

class InterruptDecision(Enum):
    CONTINUE = "continue"
    ABORT = "abort"
    MODIFY = "modify"


def agent_with_interrupts(
    plan: list[dict],
    interrupt_after: set[str] | None = None,
    simulated_human_responses: list[str] | None = None,
):
    """
    Execute a plan (list of tool calls) with interrupt points.

    Args:
        plan: List of {"tool": str, "args": dict} steps.
        interrupt_after: Tool names that trigger an interrupt after execution.
        simulated_human_responses: Pre-loaded human responses for non-interactive use.
    """
    interrupt_after = interrupt_after or set()
    responses = list(simulated_human_responses or [])
    response_idx = 0
    results = []

    for i, step in enumerate(plan):
        tool_name = step["tool"]
        tool_args = step["args"]
        print(f"\n--- Step {i+1}/{len(plan)}: {tool_name}({tool_args}) ---")

        # Execute the tool
        if tool_name in TOOLS:
            result = TOOLS[tool_name](**tool_args)
            results.append({"step": i + 1, "tool": tool_name, "result": result})
            print(f"    Result: {json.dumps(result, indent=2)}")
        else:
            print(f"    ERROR: Unknown tool '{tool_name}'")
            continue

        # Check if we should interrupt
        if tool_name in interrupt_after:
            print(f"\n    ** INTERRUPT ** Human review required after '{tool_name}'")
            if response_idx < len(responses):
                decision = responses[response_idx]
                response_idx += 1
                print(f"    Simulated human response: {decision}")
            else:
                decision = "continue"  # default fallback
                print(f"    No human input available — defaulting to: {decision}")

            if decision == "abort":
                print("    Agent ABORTED by human.")
                return results
            elif decision == "continue":
                print("    Human approved. Continuing...")
            else:
                print(f"    Human says: '{decision}' — logged for next step.")

    print("\n=== Plan execution complete ===")
    return results

In [ ]:
# ── Run the agent: interrupt after transfers, auto-continue lookups ───────

plan = [
    {"tool": "lookup_account", "args": {"account_id": "ACC-001"}},
    {"tool": "lookup_account", "args": {"account_id": "ACC-002"}},
    {"tool": "transfer_funds", "args": {"from_acct": "ACC-001", "to_acct": "ACC-002", "amount": 5000.0}},
    {"tool": "send_email", "args": {"to": "alice@example.com", "subject": "Transfer Confirmation", "body": "$5000 sent."}},
]

# Human will approve the transfer but the agent proceeds automatically for lookups
results = agent_with_interrupts(
    plan,
    interrupt_after={"transfer_funds"},
    simulated_human_responses=["continue"],
)

In [ ]:
# ── Same plan, but the human ABORTS after seeing the transfer ────────────

print("Scenario: Human rejects the transfer")
print("=" * 50)

results = agent_with_interrupts(
    plan,
    interrupt_after={"transfer_funds"},
    simulated_human_responses=["abort"],
)

print(f"\nSteps completed: {len(results)} out of {len(plan)}")
print("The email was never sent because the human stopped the agent.")

---
## 4. Approval Gates

An **approval gate** is more structured than a raw interrupt. It presents the pending action, context, and options (approve / reject / modify) in a standardized format.

Think of it as a pull-request review for agent actions.

In [ ]:
# ── Approval Gate framework ───────────────────────────────────────────────

@dataclass
class ApprovalRequest:
    """A structured request for human approval."""
    request_id: str
    action: str
    details: dict
    risk_level: str  # LOW, MEDIUM, HIGH, CRITICAL
    reason: str
    timestamp: str = field(default_factory=lambda: datetime.utcnow().isoformat())


@dataclass
class ApprovalResponse:
    """A human's response to an approval request."""
    request_id: str
    decision: str  # APPROVED, REJECTED, MODIFIED
    reviewer: str
    comment: str = ""
    modifications: dict = field(default_factory=dict)


class ApprovalGate:
    """Gate that blocks agent execution until human approval is received."""

    # Actions that always require approval, regardless of other factors
    ALWAYS_REQUIRE = {"delete_data", "transfer_funds", "send_external_email", "modify_permissions"}

    # Dollar thresholds by risk level
    AMOUNT_THRESHOLDS = {"LOW": 100, "MEDIUM": 1_000, "HIGH": 10_000}

    def __init__(self):
        self.pending: dict[str, ApprovalRequest] = {}
        self.history: list[tuple[ApprovalRequest, ApprovalResponse]] = []

    def needs_approval(self, action: str, details: dict) -> bool:
        """Decide whether an action needs human approval."""
        if action in self.ALWAYS_REQUIRE:
            return True
        amount = details.get("amount", 0)
        if amount > self.AMOUNT_THRESHOLDS["LOW"]:
            return True
        return False

    def request_approval(self, action: str, details: dict, risk_level: str, reason: str) -> ApprovalRequest:
        """Create and register an approval request."""
        req = ApprovalRequest(
            request_id=f"APR-{uuid.uuid4().hex[:8].upper()}",
            action=action,
            details=details,
            risk_level=risk_level,
            reason=reason,
        )
        self.pending[req.request_id] = req
        return req

    def submit_response(self, response: ApprovalResponse) -> bool:
        """Process a human's approval response."""
        req = self.pending.pop(response.request_id, None)
        if req is None:
            print(f"No pending request with ID {response.request_id}")
            return False
        self.history.append((req, response))
        return True

    def display_request(self, req: ApprovalRequest) -> str:
        """Format an approval request for human review."""
        border = "=" * 60
        lines = [
            border,
            "  APPROVAL REQUIRED",
            border,
            f"  Request ID:  {req.request_id}",
            f"  Action:      {req.action}",
            f"  Risk Level:  {req.risk_level}",
            f"  Reason:      {req.reason}",
            f"  Timestamp:   {req.timestamp}",
            "",
            "  Details:",
        ]
        for k, v in req.details.items():
            lines.append(f"    {k}: {v}")
        lines += ["", "  Options: [APPROVE] [REJECT] [MODIFY]", border]
        return "\n".join(lines)


gate = ApprovalGate()
print("ApprovalGate initialized.")

In [ ]:
# ── Simulate an approval workflow ─────────────────────────────────────────

actions_to_process = [
    {"action": "lookup_account", "details": {"account_id": "ACC-001"}, "risk": "LOW"},
    {"action": "transfer_funds", "details": {"from": "ACC-001", "to": "ACC-002", "amount": 25_000}, "risk": "HIGH"},
    {"action": "send_external_email", "details": {"to": "vendor@external.com", "subject": "Invoice"}, "risk": "MEDIUM"},
    {"action": "update_profile", "details": {"field": "phone", "value": "555-0199"}, "risk": "LOW"},
]

# Simulated human decisions (only for actions that need approval)
simulated_decisions = [
    ApprovalResponse(request_id="", decision="APPROVED", reviewer="Manager A", comment="Verified with client."),
    ApprovalResponse(request_id="", decision="REJECTED", reviewer="Manager A", comment="Vendor not in approved list."),
]
decision_idx = 0

for item in actions_to_process:
    action, details, risk = item["action"], item["details"], item["risk"]

    if gate.needs_approval(action, details):
        req = gate.request_approval(action, details, risk, f"Agent wants to execute: {action}")
        print(gate.display_request(req))

        # Simulate human response
        if decision_idx < len(simulated_decisions):
            resp = simulated_decisions[decision_idx]
            resp.request_id = req.request_id
            decision_idx += 1
            gate.submit_response(resp)
            print(f"  >> Human decision: {resp.decision} — {resp.comment}")
            if resp.decision == "APPROVED":
                print(f"  >> Executing {action}...\n")
            else:
                print(f"  >> Skipping {action}.\n")
        else:
            print("  >> No reviewer available — action BLOCKED.\n")
    else:
        print(f"[AUTO] {action}({details}) — no approval needed, executing.\n")

In [ ]:
# ── Review the approval audit trail ───────────────────────────────────────

print("Approval Audit Trail")
print("=" * 60)
for req, resp in gate.history:
    print(f"\n  {req.request_id}")
    print(f"    Action:   {req.action}")
    print(f"    Decision: {resp.decision}")
    print(f"    Reviewer: {resp.reviewer}")
    print(f"    Comment:  {resp.comment}")

print(f"\nTotal reviewed: {len(gate.history)}")
print(f"Still pending:  {len(gate.pending)}")

---
## 5. Confidence Thresholds

Instead of hard-coding which tools need approval, we can let the agent **self-assess** its confidence. If the agent is unsure, it routes to a human.

Two approaches:
1. **Ask the LLM** to rate its confidence on a 0-1 scale.
2. **Heuristic rules** — e.g., if the query mentions money, health, or legal terms, lower confidence.

We will implement both.

In [ ]:
# ── Approach 1: LLM self-assessed confidence ──────────────────────────────

def assess_confidence_llm(task: str) -> dict:
    """Ask the LLM to assess its own confidence for a given task."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a confidence assessor. Given a task an AI agent is about to perform, "
                    "rate your confidence that the agent can do it correctly and safely on a scale "
                    "from 0.0 to 1.0. Respond with ONLY a JSON object: "
                    '{"confidence": <float>, "reasoning": "<one sentence>"}'
                ),
            },
            {"role": "user", "content": f"Task: {task}"},
        ],
        max_tokens=150,
    )
    try:
        return json.loads(response.choices[0].message.content)
    except json.JSONDecodeError:
        return {"confidence": 0.5, "reasoning": "Could not parse LLM response"}


# Test with a range of tasks
test_tasks = [
    "Schedule a meeting for tomorrow at 2pm",
    "Transfer $50,000 to an overseas account",
    "Recommend a treatment plan for chronic back pain",
    "Summarize this 3-paragraph email",
    "Draft a legally binding contract for a merger",
]

print("LLM Self-Assessed Confidence")
print("=" * 60)
for task in test_tasks:
    result = assess_confidence_llm(task)
    conf = result["confidence"]
    route = "AUTO" if conf >= 0.7 else "HUMAN REVIEW"
    print(f"\n  [{route:^13}] Confidence: {conf:.2f}")
    print(f"  Task:      {task}")
    print(f"  Reasoning: {result['reasoning']}")

In [ ]:
# ── Approach 2: Heuristic confidence scoring ──────────────────────────────

import re

# Keywords that reduce confidence, with penalty weights
RISK_KEYWORDS = {
    "delete": 0.3,
    "transfer": 0.25,
    "payment": 0.2,
    "medical": 0.3,
    "legal": 0.25,
    "contract": 0.2,
    "password": 0.3,
    "credentials": 0.3,
    "production": 0.2,
    "PII": 0.3,
    "overseas": 0.15,
    "irreversible": 0.3,
}


def assess_confidence_heuristic(task: str) -> dict:
    """Score confidence using keyword heuristics. Fast, no LLM call needed."""
    confidence = 1.0
    triggers = []

    task_lower = task.lower()
    for keyword, penalty in RISK_KEYWORDS.items():
        if keyword.lower() in task_lower:
            confidence -= penalty
            triggers.append(keyword)

    # Check for large dollar amounts
    amounts = re.findall(r"\$([\d,]+)", task)
    for amt_str in amounts:
        amt = float(amt_str.replace(",", ""))
        if amt > 10_000:
            confidence -= 0.2
            triggers.append(f"large amount: ${amt:,.0f}")

    confidence = max(0.0, min(1.0, confidence))
    return {"confidence": round(confidence, 2), "triggers": triggers}


print("Heuristic Confidence Scoring")
print("=" * 60)
for task in test_tasks:
    result = assess_confidence_heuristic(task)
    conf = result["confidence"]
    route = "AUTO" if conf >= 0.7 else "HUMAN REVIEW"
    triggers = ", ".join(result["triggers"]) if result["triggers"] else "none"
    print(f"\n  [{route:^13}] Confidence: {conf:.2f}")
    print(f"  Task:     {task}")
    print(f"  Triggers: {triggers}")

In [ ]:
# ── Confidence-based routing function ─────────────────────────────────────

def route_by_confidence(
    task: str,
    auto_threshold: float = 0.8,
    review_threshold: float = 0.5,
) -> str:
    """
    Route a task based on combined confidence score.
    Returns: 'auto', 'human_review', or 'block'
    """
    heuristic = assess_confidence_heuristic(task)
    score = heuristic["confidence"]

    if score >= auto_threshold:
        return "auto"
    elif score >= review_threshold:
        return "human_review"
    else:
        return "block"


# Demonstrate the three-tier routing
demo_tasks = [
    "Summarize today's meeting notes",
    "Send a payment of $500 to vendor",
    "Delete all production credentials and transfer $100,000 overseas",
]

print("Three-Tier Confidence Routing")
print("=" * 60)
for task in demo_tasks:
    route = route_by_confidence(task)
    label = {"auto": "EXECUTE", "human_review": "REVIEW", "block": "BLOCKED"}[route]
    print(f"\n  [{label:^8}] {task}")

---
## 6. Escalation Patterns

Real systems need **tiered escalation**: not every issue goes to the same person.

| Tier | Handler | Example |
|------|---------|--------|
| 0 | Agent (auto) | Look up a FAQ answer |
| 1 | Junior reviewer | Approve a $500 refund |
| 2 | Senior reviewer | Approve a $50K transfer |
| 3 | Manager / compliance | Actions involving PII, legal, regulatory |

Below we implement a configurable escalation chain.

In [ ]:
# ── Escalation Engine ─────────────────────────────────────────────────────

@dataclass
class EscalationTier:
    """Defines a tier in the escalation chain."""
    level: int
    name: str
    handler: str
    max_amount: float          # max dollar amount this tier can approve
    allowed_actions: set       # actions this tier can approve
    response_time_sla: str     # expected response time


class EscalationEngine:
    """Routes actions to the appropriate approval tier."""

    def __init__(self, tiers: list[EscalationTier]):
        self.tiers = sorted(tiers, key=lambda t: t.level)
        self.escalation_log: list[dict] = []

    def determine_tier(self, action: str, amount: float = 0, tags: set = None) -> EscalationTier:
        """Find the lowest tier that can handle this action."""
        tags = tags or set()

        # Compliance tags always go to highest tier
        if tags & {"pii", "legal", "regulatory"}:
            return self.tiers[-1]

        for tier in self.tiers:
            if action in tier.allowed_actions and amount <= tier.max_amount:
                return tier

        # Default to highest tier if nothing else matches
        return self.tiers[-1]

    def escalate(self, action: str, details: dict) -> dict:
        """Escalate an action and return the routing decision."""
        amount = details.get("amount", 0)
        tags = set(details.get("tags", []))

        tier = self.determine_tier(action, amount, tags)

        record = {
            "action": action,
            "details": details,
            "routed_to": tier.name,
            "handler": tier.handler,
            "tier_level": tier.level,
            "sla": tier.response_time_sla,
            "timestamp": datetime.utcnow().isoformat(),
        }
        self.escalation_log.append(record)
        return record


# Define the tiers
tiers = [
    EscalationTier(
        level=0, name="Agent Auto", handler="system",
        max_amount=100, allowed_actions={"lookup", "summarize", "schedule"},
        response_time_sla="instant",
    ),
    EscalationTier(
        level=1, name="Junior Reviewer", handler="support-team@company.com",
        max_amount=5_000, allowed_actions={"refund", "transfer_funds", "send_email"},
        response_time_sla="15 minutes",
    ),
    EscalationTier(
        level=2, name="Senior Reviewer", handler="senior-ops@company.com",
        max_amount=100_000, allowed_actions={"refund", "transfer_funds", "modify_permissions"},
        response_time_sla="1 hour",
    ),
    EscalationTier(
        level=3, name="Compliance / Manager", handler="compliance@company.com",
        max_amount=float("inf"), allowed_actions={"all"},
        response_time_sla="4 hours",
    ),
]

engine = EscalationEngine(tiers)
print("Escalation engine ready with", len(tiers), "tiers.")

In [ ]:
# ── Test escalation routing ───────────────────────────────────────────────

test_actions = [
    {"action": "lookup", "details": {"query": "order status"}},
    {"action": "refund", "details": {"amount": 250, "order_id": "ORD-1234"}},
    {"action": "transfer_funds", "details": {"amount": 75_000, "to": "ACC-999"}},
    {"action": "refund", "details": {"amount": 500, "tags": ["pii"], "note": "Customer shared SSN in chat"}},
    {"action": "modify_permissions", "details": {"amount": 0, "user": "admin", "tags": ["regulatory"]}},
]

print("Escalation Routing Results")
print("=" * 70)
for item in test_actions:
    result = engine.escalate(item["action"], item["details"])
    print(f"\n  Action:    {result['action']}")
    print(f"  Routed to: {result['routed_to']} (Tier {result['tier_level']})")
    print(f"  Handler:   {result['handler']}")
    print(f"  SLA:       {result['sla']}")

---
## 7. UX for Human Review

A good review interface makes it easy for humans to:
1. **Understand** what the agent wants to do and why.
2. **Decide** quickly (approve / reject / modify).
3. **Provide feedback** that the agent can learn from.

Below we build a text-based review console that could be adapted to a web UI.

In [ ]:
# ── Human Review Console (text-based) ─────────────────────────────────────

class ReviewConsole:
    """Text-based review interface for human-in-the-loop workflows."""

    def __init__(self):
        self.review_queue: list[dict] = []
        self.completed_reviews: list[dict] = []

    def add_to_queue(self, item: dict):
        """Add an item to the review queue."""
        item["queued_at"] = datetime.utcnow().isoformat()
        item["status"] = "pending"
        self.review_queue.append(item)

    def render_review_item(self, item: dict) -> str:
        """Render a single review item as a formatted text card."""
        lines = []
        lines.append("+" + "-" * 58 + "+")
        lines.append(f"| {'REVIEW ITEM':^56} |")
        lines.append("+" + "-" * 58 + "+")
        lines.append(f"| Action:     {item.get('action', 'N/A'):<44} |")
        lines.append(f"| Risk:       {item.get('risk_level', 'N/A'):<44} |")
        lines.append(f"| Queued:     {item.get('queued_at', 'N/A')[:19]:<44} |")
        lines.append("+" + "-" * 58 + "+")

        # Agent's reasoning
        reasoning = item.get("agent_reasoning", "No reasoning provided.")
        lines.append("| Agent reasoning:" + " " * 40 + "|")
        for chunk in textwrap.wrap(reasoning, width=54):
            lines.append(f"|   {chunk:<55}|")

        # Details
        lines.append("+" + "-" * 58 + "+")
        lines.append("| Details:" + " " * 49 + "|")
        for k, v in item.get("details", {}).items():
            entry = f"{k}: {v}"
            lines.append(f"|   {entry:<55}|")

        lines.append("+" + "-" * 58 + "+")
        lines.append(f"| Options: [A]pprove  [R]eject  [M]odify  [S]kip{' ' * 10}|")
        lines.append("+" + "-" * 58 + "+")
        return "\n".join(lines)

    def process_queue(self, simulated_inputs: list[str]):
        """Process the review queue with simulated human inputs."""
        input_idx = 0
        while self.review_queue:
            item = self.review_queue.pop(0)
            print(self.render_review_item(item))

            if input_idx < len(simulated_inputs):
                choice = simulated_inputs[input_idx]
                input_idx += 1
            else:
                choice = "S"  # skip if no more inputs

            decision_map = {"A": "APPROVED", "R": "REJECTED", "M": "MODIFIED", "S": "SKIPPED"}
            decision = decision_map.get(choice.upper(), "SKIPPED")
            item["status"] = decision
            item["reviewed_at"] = datetime.utcnow().isoformat()
            self.completed_reviews.append(item)
            print(f"\n  >> Decision: {decision}\n")


console = ReviewConsole()
print("ReviewConsole ready.")

In [ ]:
# ── Populate and process the review queue ─────────────────────────────────

console.add_to_queue({
    "action": "transfer_funds",
    "risk_level": "HIGH",
    "agent_reasoning": "Customer requested transfer of $25,000 to account ACC-002. "
                       "Verified customer identity via two-factor auth. Amount exceeds "
                       "auto-approval threshold.",
    "details": {"from": "ACC-001", "to": "ACC-002", "amount": "$25,000"},
})

console.add_to_queue({
    "action": "delete_user_data",
    "risk_level": "CRITICAL",
    "agent_reasoning": "GDPR deletion request received. Customer wants all personal data "
                       "removed. This is irreversible and affects 3 linked services.",
    "details": {"user_id": "USR-4421", "scope": "all_personal_data", "services": "3"},
})

console.add_to_queue({
    "action": "send_external_email",
    "risk_level": "MEDIUM",
    "agent_reasoning": "Agent drafted a follow-up email to an external vendor. "
                       "Contains pricing information that may be confidential.",
    "details": {"to": "vendor@partner.com", "subject": "Q4 Pricing Update"},
})

# Simulated human decisions: Approve transfer, Reject deletion, Approve email
console.process_queue(["A", "R", "A"])

---
## 8. Resuming After Interruption

When an agent pauses for human input, it must **save its state** so it can resume exactly where it left off. This is critical for:
- Long-running workflows (hours or days between human reviews)
- Multi-step plans where later steps depend on earlier results
- Crash recovery

We will build a checkpoint-based state manager.

In [ ]:
# ── Agent State Manager ───────────────────────────────────────────────────

@dataclass
class AgentCheckpoint:
    """Snapshot of agent state at an interrupt point."""
    checkpoint_id: str
    step_index: int
    plan: list[dict]
    completed_results: list[dict]
    pending_action: dict | None
    context: dict  # any additional context the agent needs
    created_at: str = field(default_factory=lambda: datetime.utcnow().isoformat())
    status: str = "paused"  # paused, resumed, completed, aborted


class StatefulAgent:
    """Agent that can pause, save state, and resume."""

    def __init__(self, tools: dict):
        self.tools = tools
        self.checkpoints: dict[str, AgentCheckpoint] = {}

    def execute_plan(
        self,
        plan: list[dict],
        interrupt_before: set[str] | None = None,
        context: dict | None = None,
    ) -> tuple[list[dict], str | None]:
        """Execute a plan, pausing before specified actions. Returns (results, checkpoint_id)."""
        interrupt_before = interrupt_before or set()
        context = context or {}
        results = []

        for i, step in enumerate(plan):
            tool_name = step["tool"]

            # Check for interrupt BEFORE executing
            if tool_name in interrupt_before:
                checkpoint = AgentCheckpoint(
                    checkpoint_id=f"CKP-{uuid.uuid4().hex[:8].upper()}",
                    step_index=i,
                    plan=plan,
                    completed_results=results.copy(),
                    pending_action=step,
                    context=context,
                )
                self.checkpoints[checkpoint.checkpoint_id] = checkpoint
                print(f"\n  [PAUSED] Checkpoint {checkpoint.checkpoint_id} created at step {i+1}")
                print(f"  Pending action: {tool_name}({step['args']})")
                print(f"  Completed {len(results)} of {len(plan)} steps so far.")
                return results, checkpoint.checkpoint_id

            # Execute the step
            print(f"  [Step {i+1}] {tool_name}({step['args']})")
            if tool_name in self.tools:
                result = self.tools[tool_name](**step["args"])
                results.append({"step": i + 1, "tool": tool_name, "result": result})
                print(f"    -> {result}")
            else:
                print(f"    -> ERROR: unknown tool")

        print(f"\n  [DONE] Plan completed. {len(results)} steps executed.")
        return results, None

    def resume(self, checkpoint_id: str, human_decision: str = "approve") -> tuple[list[dict], str | None]:
        """Resume from a checkpoint after human input."""
        ckp = self.checkpoints.get(checkpoint_id)
        if ckp is None:
            print(f"Checkpoint {checkpoint_id} not found.")
            return [], None

        print(f"\n  [RESUMING] Checkpoint {checkpoint_id}")
        print(f"  Human decision: {human_decision}")

        if human_decision == "reject":
            ckp.status = "aborted"
            print("  Agent ABORTED by human. Returning partial results.")
            return ckp.completed_results, None

        ckp.status = "resumed"

        # Continue from the paused step
        remaining_plan = ckp.plan[ckp.step_index:]
        results = ckp.completed_results.copy()

        for i, step in enumerate(remaining_plan):
            actual_step = ckp.step_index + i
            tool_name = step["tool"]
            print(f"  [Step {actual_step+1}] {tool_name}({step['args']})")

            if tool_name in self.tools:
                result = self.tools[tool_name](**step["args"])
                results.append({"step": actual_step + 1, "tool": tool_name, "result": result})
                print(f"    -> {result}")

        ckp.status = "completed"
        print(f"\n  [DONE] Resumed plan completed. {len(results)} total steps.")
        return results, None


agent = StatefulAgent(TOOLS)
print("StatefulAgent ready.")

In [ ]:
# ── Execute a plan that pauses before the transfer step ───────────────────

plan = [
    {"tool": "lookup_account", "args": {"account_id": "ACC-001"}},
    {"tool": "lookup_account", "args": {"account_id": "ACC-002"}},
    {"tool": "transfer_funds", "args": {"from_acct": "ACC-001", "to_acct": "ACC-002", "amount": 5000.0}},
    {"tool": "send_email", "args": {"to": "alice@example.com", "subject": "Transfer Done", "body": "$5000 sent."}},
]

print("Phase 1: Execute until interrupt")
print("=" * 50)
results, ckp_id = agent.execute_plan(plan, interrupt_before={"transfer_funds"})
print(f"\nCheckpoint ID: {ckp_id}")
print(f"Results so far: {len(results)} steps")

In [ ]:
# ── Simulate time passing... then human approves ──────────────────────────

print("--- Some time passes (human reviews the pending action) ---\n")

print("Phase 2: Resume after approval")
print("=" * 50)
final_results, _ = agent.resume(ckp_id, human_decision="approve")
print(f"\nFinal results: {len(final_results)} steps completed")
for r in final_results:
    print(f"  Step {r['step']}: {r['tool']} -> {r['result']}")

In [ ]:
# ── Demonstrate rejection: run the same plan, but reject this time ────────

print("Phase 1 (again): Execute until interrupt")
print("=" * 50)
results2, ckp_id2 = agent.execute_plan(plan, interrupt_before={"transfer_funds"})

print("\nPhase 2: Human REJECTS")
print("=" * 50)
final2, _ = agent.resume(ckp_id2, human_decision="reject")

print(f"\nOnly {len(final2)} steps completed — transfer and email were never executed.")

---
## 9. LangGraph `interrupt()`

LangGraph provides a first-class `interrupt()` primitive that pauses graph execution and waits for human input. This is the production-grade version of what we built by hand.

Key concepts:
- **`interrupt(value)`** — pauses the graph node, sends `value` to the caller
- **`Command(resume=...)`** — caller sends a response that resumes the graph
- **Checkpointer** — persists graph state so it survives restarts

Let's build a financial approval workflow using LangGraph.

In [ ]:
# ── LangGraph interrupt() demo ────────────────────────────────────────────

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from typing import TypedDict

print("LangGraph imports successful.")

In [ ]:
# ── Define the graph state ────────────────────────────────────────────────

class TransactionState(TypedDict):
    request: dict           # the incoming transaction request
    risk_assessment: str    # LOW / MEDIUM / HIGH / CRITICAL
    human_decision: str     # approved / rejected / modified
    result: str             # final outcome


# ── Node functions ────────────────────────────────────────────────────────

def assess_risk(state: TransactionState) -> dict:
    """Assess the risk level of a transaction."""
    request = state["request"]
    amount = request.get("amount", 0)

    if amount > 50_000:
        risk = "CRITICAL"
    elif amount > 10_000:
        risk = "HIGH"
    elif amount > 1_000:
        risk = "MEDIUM"
    else:
        risk = "LOW"

    print(f"  [assess_risk] Amount: ${amount:,.2f} -> Risk: {risk}")
    return {"risk_assessment": risk}


def human_review(state: TransactionState) -> dict:
    """Pause for human review using LangGraph interrupt()."""
    request = state["request"]
    risk = state["risk_assessment"]

    # This is where the magic happens: interrupt() pauses the graph
    decision = interrupt({
        "message": "Human review required",
        "transaction": request,
        "risk_level": risk,
        "options": ["approve", "reject", "modify"],
    })

    print(f"  [human_review] Received decision: {decision}")
    return {"human_decision": decision}


def execute_transaction(state: TransactionState) -> dict:
    """Execute or reject the transaction based on human decision."""
    decision = state["human_decision"]
    request = state["request"]

    if decision == "approve":
        txn_id = f"TXN-{uuid.uuid4().hex[:8].upper()}"
        result = f"EXECUTED: ${request['amount']:,.2f} transferred. Confirmation: {txn_id}"
    elif decision == "reject":
        result = "REJECTED: Transaction was declined by reviewer."
    else:
        result = f"MODIFIED: Human requested changes — {decision}"

    print(f"  [execute_transaction] {result}")
    return {"result": result}


# ── Routing function ──────────────────────────────────────────────────────

def route_by_risk(state: TransactionState) -> str:
    """Route to human review if risk is above LOW."""
    if state["risk_assessment"] == "LOW":
        print("  [router] Low risk — auto-approving.")
        return "auto_approve"
    else:
        print("  [router] Elevated risk — routing to human review.")
        return "human_review"


def auto_approve(state: TransactionState) -> dict:
    """Auto-approve low-risk transactions."""
    print("  [auto_approve] Transaction auto-approved.")
    return {"human_decision": "approve"}


print("Node functions defined.")

In [ ]:
# ── Build the graph ───────────────────────────────────────────────────────

builder = StateGraph(TransactionState)

# Add nodes
builder.add_node("assess_risk", assess_risk)
builder.add_node("human_review", human_review)
builder.add_node("auto_approve", auto_approve)
builder.add_node("execute_transaction", execute_transaction)

# Add edges
builder.add_edge(START, "assess_risk")
builder.add_conditional_edges("assess_risk", route_by_risk)
builder.add_edge("human_review", "execute_transaction")
builder.add_edge("auto_approve", "execute_transaction")
builder.add_edge("execute_transaction", END)

# Compile with a memory checkpointer (required for interrupt)
checkpointer = MemorySaver()
graph = builder.compile(checkpointer=checkpointer)

print("Transaction approval graph compiled.")
print("Nodes:", [n for n in builder.nodes])
print("\nGraph visualization (if in Jupyter):")
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print("  (Install pygraphviz or use Jupyter for visual graph)")
    print("  Flow: START -> assess_risk -> [human_review | auto_approve] -> execute_transaction -> END")

In [ ]:
# ── Test 1: Low-risk transaction (auto-approved, no interrupt) ────────────

print("Test 1: Low-Risk Transaction ($50)")
print("=" * 50)

config = {"configurable": {"thread_id": "txn-low-risk"}}
input_state = {
    "request": {"from": "ACC-001", "to": "ACC-002", "amount": 50.00, "description": "Coffee reimbursement"},
    "risk_assessment": "",
    "human_decision": "",
    "result": "",
}

result = graph.invoke(input_state, config=config)
print(f"\nFinal result: {result['result']}")

In [ ]:
# ── Test 2: High-risk transaction (requires human approval) ───────────────

print("Test 2: High-Risk Transaction ($25,000)")
print("=" * 50)

config_high = {"configurable": {"thread_id": "txn-high-risk"}}
input_high = {
    "request": {"from": "ACC-001", "to": "ACC-999", "amount": 25_000.00, "description": "Vendor payment"},
    "risk_assessment": "",
    "human_decision": "",
    "result": "",
}

# Phase 1: run until interrupt
result = graph.invoke(input_high, config=config_high)
print(f"\nGraph paused. State so far: risk={result.get('risk_assessment', '?')}")
print("Waiting for human input...")

In [ ]:
# ── Phase 2: Human approves, graph resumes ───────────────────────────────

print("Human approves the transaction...")
print("=" * 50)

# Resume with Command(resume=<value>)
result = graph.invoke(Command(resume="approve"), config=config_high)
print(f"\nFinal result: {result['result']}")

In [ ]:
# ── Test 3: Critical transaction — human REJECTS ──────────────────────────

print("Test 3: Critical Transaction ($75,000) — REJECTED")
print("=" * 50)

config_crit = {"configurable": {"thread_id": "txn-critical"}}
input_crit = {
    "request": {"from": "ACC-002", "to": "ACC-EXT-777", "amount": 75_000.00, "description": "Suspicious transfer"},
    "risk_assessment": "",
    "human_decision": "",
    "result": "",
}

# Phase 1: runs until interrupt
result = graph.invoke(input_crit, config=config_crit)
print(f"Graph paused. Risk: {result.get('risk_assessment', '?')}")

# Phase 2: human rejects
print("\nHuman REJECTS the transaction...")
result = graph.invoke(Command(resume="reject"), config=config_crit)
print(f"\nFinal result: {result['result']}")

### What just happened

1. The graph ran `assess_risk`, which classified the transaction.
2. The conditional edge routed to `human_review` (for HIGH/CRITICAL) or `auto_approve` (for LOW).
3. `human_review` called `interrupt()`, which **paused the entire graph** and returned control to us.
4. We sent `Command(resume="approve")` or `Command(resume="reject")`, which **injected the human's decision** and resumed the graph.
5. `execute_transaction` used the decision to produce the final result.

The `MemorySaver` checkpointer kept the full graph state between calls. In production, you would use `SqliteSaver` or `PostgresSaver` for durability.

---
## 10. Exercises

### Exercise 1 — Conceptual (10 min)

Answer in markdown cells below:

1. **When should you NOT use human-in-the-loop?** Give three examples of tasks where adding human review would reduce value rather than add it.

2. **Alert fatigue:** If every action pings a human, reviewers start rubber-stamping. Describe two strategies to keep signal-to-noise high in an approval queue.

3. **Latency trade-off:** A customer-facing chatbot adds a 2-minute human approval step for refunds over $100. What is the UX impact? How would you mitigate it?

**Your answers here:**

1. ...

2. ...

3. ...

### Exercise 2 — Coding: Build an Approval Workflow (30 min)

Build a LangGraph workflow for a **content moderation agent** that:

1. Receives a user-generated post (text).
2. Uses an LLM to classify the post: `safe`, `borderline`, or `violation`.
3. If `safe` — auto-publishes.
4. If `borderline` — uses `interrupt()` to send the post to a human moderator.
5. If `violation` — auto-rejects with a reason.
6. Human moderator can: `publish`, `reject`, or `edit` (provide modified text).

**Requirements:**
- Use `StateGraph` with a `MemorySaver` checkpointer.
- Include at least 4 nodes: `classify`, `human_moderate`, `auto_publish`, `auto_reject`.
- Test with three posts: one safe, one borderline, one violation.

In [ ]:
# ── Exercise 2: Your code here ────────────────────────────────────────────

# Starter scaffold:

class ModerationState(TypedDict):
    post_text: str
    classification: str   # safe / borderline / violation
    reason: str
    moderator_decision: str
    final_status: str     # published / rejected / edited


def classify_post(state: ModerationState) -> dict:
    # TODO: Use the LLM to classify the post
    pass


def human_moderate(state: ModerationState) -> dict:
    # TODO: Use interrupt() to pause for human review
    pass


# Build and test your graph below
# ...

### Exercise 3 — Design: HITL for a Financial Trading Agent (20 min)

You are designing a human-in-the-loop system for an AI-powered **financial trading agent** that can:
- Monitor market conditions in real-time
- Place buy/sell orders for stocks, bonds, and options
- Rebalance portfolios
- Execute stop-loss triggers

**Design document requirements:**

1. **Risk classification matrix:** Define at least 4 risk tiers with specific criteria (dollar amounts, asset types, market conditions).

2. **Escalation chain:** Who reviews what? Include at least 3 tiers (e.g., automated, trader, risk manager, compliance officer).

3. **Time-sensitive handling:** Markets move fast. How do you handle the case where a human takes too long to respond? (Hint: timeouts, fallback strategies, price guards.)

4. **Audit trail:** What information must be logged for regulatory compliance?

5. **Edge case:** The agent detects a flash crash and wants to sell all holdings immediately. The human reviewer is at lunch. What happens?

Write your design below as structured markdown.

**Your design here:**

#### 1. Risk Classification Matrix

| Tier | Criteria | Example |
|------|----------|---------|
| ... | ... | ... |

#### 2. Escalation Chain

...

#### 3. Time-Sensitive Handling

...

#### 4. Audit Trail

...

#### 5. Flash Crash Edge Case

...

---
## Summary

| Concept | What You Built |
|---------|---------------|
| **Interrupt Points** | Agent loop that pauses at configurable steps |
| **Approval Gates** | Structured approve/reject/modify framework with audit trail |
| **Confidence Thresholds** | LLM self-assessment + heuristic scoring for routing decisions |
| **Escalation Patterns** | Tiered escalation engine with SLAs and compliance routing |
| **Review UX** | Text-based review console with queue management |
| **State Persistence** | Checkpoint-based pause/resume for long-running workflows |
| **LangGraph `interrupt()`** | Production-grade HITL using LangGraph's built-in primitives |

### Key takeaways

1. **Not every action needs a human.** Classify risk and route only what matters.
2. **Structured approvals beat ad-hoc interrupts.** Give reviewers context, options, and audit trails.
3. **Confidence is a spectrum.** Use thresholds to create auto/review/block tiers.
4. **State must survive interruptions.** Use checkpointers to persist and resume.
5. **LangGraph `interrupt()` is the production primitive.** It handles state, serialization, and resumption.